# 02 Data Cleaning and Preparation

This notebook cleans and prepares the selected Ministry of Health New Zealand Health Survey datasets for PostgreSQL and Power BI reporting.

The project focuses on seven health access barrier indicators identified during data profiling.

In [1]:
import pandas as pd
from pathlib import Path

# Project folders
raw_path = Path("../data/raw")
config_path = Path("../config")
processed_path = Path("../data/processed")

# Load raw datasets
prevalence = pd.read_csv(raw_path / "prevalence_mean.csv")
comparison = pd.read_csv(raw_path / "subgroup_comparison.csv")
trend = pd.read_csv(raw_path / "changes_over_time.csv")

# Load indicator mapping created in the profiling step
indicator_mapping = pd.read_csv(config_path / "indicator_mapping.csv")

print("Raw datasets and indicator mapping loaded successfully.")

Raw datasets and indicator mapping loaded successfully.


## Load Selected Indicators

The indicator mapping table defines the seven health access barrier indicators included in this project.

These indicators are used to filter the raw datasets before cleaning and modelling.

In [2]:
selected_indicators = indicator_mapping.loc[
    indicator_mapping["include_flag"] == True,
    "indicator_name"
].tolist()

selected_indicators

['Unfilled prescription due to cost',
 'Unmet need for GP due to cost',
 'Unmet need for GP due to transport',
 'Unmet need for GP due to wait time',
 'Unmet need for GP due to work',
 'Unmet need for dental health care due to cost',
 'Unmet need for mental health care and addictions services']

## Prepare Prevalence Fact Table

The prevalence dataset shows the percentage or mean value for each indicator, year and population group.

For this project, the table is filtered to the selected health access barrier indicators and cleaned into a reporting-ready fact table.

In [3]:
# Filter prevalence dataset to selected health access indicators

prevalence_filtered = prevalence[
    prevalence["short.description"].isin(selected_indicators)
].copy()

prevalence_filtered.shape

(2600, 19)

In [4]:
prevalence_filtered.head()

,population,short.description,year,group,total,flag_for_publishing,total.low.CI,total.high.CI,male,male_flag_for_publishing,male.low.CI,male.high.CI,female,female_flag_for_publishing,female.low.CI,female.high.CI,estimated.number,estimated.number.low.CI,estimated.number.high.CI
40729,adults,Unmet need for GP due to cost,2011,Total,13.6,NaN,12.6,14.5,9.6,NaN,8.6,10.7,17.3,NaN,15.9,18.6,474000.0,441000.0,506000.0
40730,adults,Unmet need for GP due to cost,2012,Total,14.6,NaN,13.8,15.4,10.9,NaN,9.9,12.1,18.0,NaN,16.9,19.1,513000.0,485000.0,542000.0
40731,adults,Unmet need for GP due to cost,2013,Total,14.1,NaN,13.3,15.0,11.0,NaN,9.9,12.3,17.0,NaN,15.7,18.3,505000.0,474000.0,535000.0
40732,adults,Unmet need for GP due to cost,2014,Total,13.8,NaN,13.0,14.6,10.3,NaN,9.4,11.3,17.1,NaN,16.0,18.2,505000.0,477000.0,533000.0
40733,adults,Unmet need for GP due to cost,2015,Total,14.3,NaN,13.5,15.1,10.4,NaN,9.4,11.5,18.0,NaN,16.8,19.2,537000.0,508000.0,567000.0


## Standardise Prevalence Table

The filtered prevalence table is cleaned by renaming columns and joining the indicator mapping table.

This creates a clearer table for PostgreSQL and Power BI reporting.

In [5]:
# Rename columns for easier use in PostgreSQL and Power BI

fact_prevalence = prevalence_filtered.rename(columns={
    "short.description": "indicator_name",
    "total": "total_value",
    "total.low.CI": "total_low_ci",
    "total.high.CI": "total_high_ci",
    "male": "male_value",
    "male.low.CI": "male_low_ci",
    "male.high.CI": "male_high_ci",
    "female": "female_value",
    "female.low.CI": "female_low_ci",
    "female.high.CI": "female_high_ci",
    "estimated.number": "estimated_number",
    "estimated.number.low.CI": "estimated_number_low_ci",
    "estimated.number.high.CI": "estimated_number_high_ci"
})

fact_prevalence.head()

,population,indicator_name,year,group,total_value,flag_for_publishing,total_low_ci,total_high_ci,male_value,male_flag_for_publishing,male_low_ci,male_high_ci,female_value,female_flag_for_publishing,female_low_ci,female_high_ci,estimated_number,estimated_number_low_ci,estimated_number_high_ci
40729,adults,Unmet need for GP due to cost,2011,Total,13.6,NaN,12.6,14.5,9.6,NaN,8.6,10.7,17.3,NaN,15.9,18.6,474000.0,441000.0,506000.0
40730,adults,Unmet need for GP due to cost,2012,Total,14.6,NaN,13.8,15.4,10.9,NaN,9.9,12.1,18.0,NaN,16.9,19.1,513000.0,485000.0,542000.0
40731,adults,Unmet need for GP due to cost,2013,Total,14.1,NaN,13.3,15.0,11.0,NaN,9.9,12.3,17.0,NaN,15.7,18.3,505000.0,474000.0,535000.0
40732,adults,Unmet need for GP due to cost,2014,Total,13.8,NaN,13.0,14.6,10.3,NaN,9.4,11.3,17.1,NaN,16.0,18.2,505000.0,477000.0,533000.0
40733,adults,Unmet need for GP due to cost,2015,Total,14.3,NaN,13.5,15.1,10.4,NaN,9.4,11.5,18.0,NaN,16.8,19.2,537000.0,508000.0,567000.0


In [48]:
fact_prevalence.to_csv(
    processed_path / "fact_prevalence.csv",
    index=False
)

print("fact_prevalence.csv saved successfully.")

fact_prevalence.csv saved successfully.


### Filter to Selected Health Access Indicators

The subgroup comparison dataset contains many health indicators.  
Only the seven selected health access indicators are kept for this project.

This ensures that subgroup equity analysis uses the same indicator scope as the prevalence analysis.

In [6]:
comparison_filtered = comparison[
    comparison["short.description"].isin(selected_indicators)
].copy()

comparison_filtered.shape

(186, 8)

### Standardise Column Names

Column names are standardised to make the table easier to use in PostgreSQL, SQL queries and Power BI.

The original dataset uses names such as `adjusted.rate.ratio`; these are renamed to snake_case names such as `adjusted_rate_ratio`.

In [7]:
fact_subgroup_comparison = comparison_filtered.rename(columns={
    "short.description": "indicator_name",
    "adjusted.rate.ratio": "adjusted_rate_ratio",
    "adjusted.rate.ratio.low.CI": "adjusted_rate_ratio_low_ci",
    "adjusted.rate.ratio.high.CI": "adjusted_rate_ratio_high_ci",
    "adjusted.for": "adjusted_for"
})

fact_subgroup_comparison.head()

,population,indicator_name,year,comparison,adjusted_rate_ratio,adjusted_rate_ratio_low_ci,adjusted_rate_ratio_high_ci,adjusted_for
711,children,Unmet need for GP due to cost,2024,Boys vs girls,0.63,0.32,1.23,Age
712,children,Unmet need for GP due to cost,2024,Māori vs non-Māori,1.24,0.64,2.40,"Age, gender"
713,children,Unmet need for GP due to cost,2024,Māori boys vs non-Māori boys,0.70,0.25,1.99,Age
714,children,Unmet need for GP due to cost,2024,Māori girls vs non-Māori girls,1.70,0.61,4.72,Age
715,children,Unmet need for GP due to cost,2024,Pacific vs non-Pacific,3.39,1.43,8.04,"Age, gender"


### Add Indicator Metadata

The indicator mapping table is joined to add indicator IDs, short names, theme and barrier type.

This supports a cleaner reporting model and makes the table easier to connect with other fact tables in PostgreSQL and Power BI.

In [8]:
fact_subgroup_comparison = fact_subgroup_comparison.merge(
    indicator_mapping[
        [
            "indicator_id",
            "indicator_name",
            "indicator_short_name",
            "theme",
            "barrier_type"
        ]
    ],
    on="indicator_name",
    how="left"
)

fact_subgroup_comparison.head()

,population,indicator_name,year,comparison,adjusted_rate_ratio,adjusted_rate_ratio_low_ci,adjusted_rate_ratio_high_ci,adjusted_for,indicator_id,indicator_short_name,theme,barrier_type
0,children,Unmet need for GP due to cost,2024,Boys vs girls,0.63,0.32,1.23,Age,2,GP cost barrier,Health Access,Cost
1,children,Unmet need for GP due to cost,2024,Māori vs non-Māori,1.24,0.64,2.40,"Age, gender",2,GP cost barrier,Health Access,Cost
2,children,Unmet need for GP due to cost,2024,Māori boys vs non-Māori boys,0.70,0.25,1.99,Age,2,GP cost barrier,Health Access,Cost
3,children,Unmet need for GP due to cost,2024,Māori girls vs non-Māori girls,1.70,0.61,4.72,Age,2,GP cost barrier,Health Access,Cost
4,children,Unmet need for GP due to cost,2024,Pacific vs non-Pacific,3.39,1.43,8.04,"Age, gender",2,GP cost barrier,Health Access,Cost


In [9]:
fact_subgroup_comparison = fact_subgroup_comparison[
    [
        "indicator_id",
        "indicator_name",
        "indicator_short_name",
        "theme",
        "barrier_type",
        "population",
        "year",
        "comparison",
        "adjusted_rate_ratio",
        "adjusted_rate_ratio_low_ci",
        "adjusted_rate_ratio_high_ci",
        "adjusted_for"
    ]
]

fact_subgroup_comparison.head()

,indicator_id,indicator_name,indicator_short_name,theme,barrier_type,population,year,comparison,adjusted_rate_ratio,adjusted_rate_ratio_low_ci,adjusted_rate_ratio_high_ci,adjusted_for
0,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,2024,Boys vs girls,0.63,0.32,1.23,Age
1,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,2024,Māori vs non-Māori,1.24,0.64,2.40,"Age, gender"
2,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,2024,Māori boys vs non-Māori boys,0.70,0.25,1.99,Age
3,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,2024,Māori girls vs non-Māori girls,1.70,0.61,4.72,Age
4,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,2024,Pacific vs non-Pacific,3.39,1.43,8.04,"Age, gender"


In [10]:
print("Original subgroup comparison rows:", comparison.shape[0])
print("Filtered subgroup comparison rows:", comparison_filtered.shape[0])
print("Cleaned fact_subgroup_comparison rows:", fact_subgroup_comparison.shape[0])

Original subgroup comparison rows: 3198
Filtered subgroup comparison rows: 186
Cleaned fact_subgroup_comparison rows: 186


### Validate Subgroup Comparison Table

The processed table is validated before saving.

The validation checks whether:
- row counts are preserved after filtering and joining
- all records match the indicator mapping table
- only the seven selected indicators are included
- key fields are not missing
- duplicate comparison records are not created
- numeric fields have suitable data types

In [11]:
print("Missing indicator_id:", fact_subgroup_comparison["indicator_id"].isna().sum())
print("Number of indicators:", fact_subgroup_comparison["indicator_name"].nunique())
print("Year range:", fact_subgroup_comparison["year"].min(), "to", fact_subgroup_comparison["year"].max())

Missing indicator_id: 0
Number of indicators: 7
Year range: 2024 to 2024


In [12]:
key_columns = [
    "indicator_id",
    "indicator_name",
    "population",
    "year",
    "comparison",
    "adjusted_rate_ratio"
]

fact_subgroup_comparison[key_columns].isna().sum()

indicator_id           0
indicator_name         0
population             0
year                   0
comparison             0
adjusted_rate_ratio    0
dtype: int64

In [13]:
duplicate_count = fact_subgroup_comparison.duplicated(
    subset=[
        "indicator_id",
        "population",
        "year",
        "comparison",
        "adjusted_for"
    ]
).sum()

duplicate_count

0

In [14]:
fact_subgroup_comparison["comparison"].drop_duplicates().sort_values()

8                           Asian boys vs non-Asian boys
9                         Asian girls vs non-Asian girls
96                            Asian men vs non-Asian men
7                                     Asian vs non-Asian
97                        Asian women vs non-Asian women
0                                          Boys vs girls
101               Disabled adults vs non-disabled adults
11     Disabled children vs non-disabled children (5-14)
88                                          Men vs women
10                       Most deprived vs least deprived
33                Most deprived vs least deprived - boys
47               Most deprived vs least deprived - girls
99                 Most deprived vs least deprived - men
100              Most deprived vs least deprived - women
2                           Māori boys vs non-Māori boys
3                         Māori girls vs non-Māori girls
90                            Māori men vs non-Māori men
1                              

In [15]:
fact_subgroup_comparison[
    [
        "adjusted_rate_ratio",
        "adjusted_rate_ratio_low_ci",
        "adjusted_rate_ratio_high_ci"
    ]
].dtypes

adjusted_rate_ratio            float64
adjusted_rate_ratio_low_ci     float64
adjusted_rate_ratio_high_ci    float64
dtype: object

### Save Processed Subgroup Comparison Table

After validation, the cleaned table is saved to the processed data folder as `fact_subgroup_comparison.csv`.

In [16]:
fact_subgroup_comparison.to_csv(
    processed_path / "fact_subgroup_comparison.csv",
    index=False
)

print("fact_subgroup_comparison.csv saved successfully.")

fact_subgroup_comparison.csv saved successfully.


## Build Trend Fact Table

This section creates `fact_trend` from the raw changes over time dataset.

The trend dataset stores yearly percentage values in separate columns, such as `percent.11`, `percent.12` and `percent.24`.

For reporting in PostgreSQL and Power BI, these year columns are reshaped into a long format where each row represents one indicator, one population group and one year.

This table supports trend analysis, showing whether selected health access barriers have improved or worsened over time.

### Filter to Selected Health Access Indicators

The same seven selected health access indicators are applied to the trend dataset.

This keeps the trend analysis consistent with the prevalence and subgroup comparison tables.

In [17]:
trend_filtered = trend[
    trend["short.description"].isin(selected_indicators)
].copy()

trend_filtered.shape

(532, 31)

### Reshape Year Columns from Wide to Long Format

The raw trend table stores each year as a separate column.

The `melt()` function is used to convert these year columns into rows, so the table becomes easier to query in SQL and easier to visualise in Power BI.

In [18]:
percent_columns = [
    col for col in trend_filtered.columns
    if col.startswith("percent.")
]

percent_columns

['percent.11',
 'percent.12',
 'percent.13',
 'percent.14',
 'percent.15',
 'percent.16',
 'percent.17',
 'percent.18',
 'percent.19',
 'percent.20',
 'percent.21',
 'percent.22',
 'percent.23',
 'percent.24']

In [19]:
fact_trend = trend_filtered.melt(
    id_vars=[
        "population",
        "group",
        "short.description"
    ],
    value_vars=percent_columns,
    var_name="year_label",
    value_name="percent_raw"
)

fact_trend.head()

,population,group,short.description,year_label,percent_raw
0,children,Total,Unmet need for GP due to cost,percent.11,4.7
1,children,Boys,Unmet need for GP due to cost,percent.11,5.0
2,children,Girls,Unmet need for GP due to cost,percent.11,4.4
3,children,0-4,Unmet need for GP due to cost,percent.11,2.1
4,children,'5-9,Unmet need for GP due to cost,percent.11,6.8


### Create Year Column

The original year labels use short names such as `percent.11`.

These labels are converted into full year values such as `2011`.

In [20]:
fact_trend["year"] = (
    fact_trend["year_label"]
    .str.replace("percent.", "20", regex=False)
    .astype(int)
)

fact_trend.head()

,population,group,short.description,year_label,percent_raw,year
0,children,Total,Unmet need for GP due to cost,percent.11,4.7,2011
1,children,Boys,Unmet need for GP due to cost,percent.11,5.0,2011
2,children,Girls,Unmet need for GP due to cost,percent.11,4.4,2011
3,children,0-4,Unmet need for GP due to cost,percent.11,2.1,2011
4,children,'5-9,Unmet need for GP due to cost,percent.11,6.8,2011


### Clean Percentage Values and Publishing Flags

Some percentage values include publishing flags, such as `1.2 e` or `S`.

The numeric percentage is separated from the publishing flag:
- `1.2 e` becomes `percent_value = 1.2` and `trend_flag_for_publishing = e`
- `S` means the value is suppressed, so the numeric value remains missing

This keeps the official publishing information instead of deleting it.

In [21]:
import re
import pandas as pd

def extract_percent_value(value):
    if pd.isna(value):
        return pd.NA
    
    text = str(value).strip()
    match = re.search(r"\d+(\.\d+)?", text)
    
    if match:
        return float(match.group())
    
    return pd.NA


def extract_trend_flag(value):
    if pd.isna(value):
        return pd.NA
    
    text = str(value).strip()
    
    if text == "S":
        return "S"
    
    parts = text.split()
    
    if len(parts) > 1:
        return parts[1]
    
    return pd.NA

In [22]:
fact_trend["percent_value"] = fact_trend["percent_raw"].apply(extract_percent_value)

fact_trend["trend_flag_for_publishing"] = fact_trend["percent_raw"].apply(extract_trend_flag)

fact_trend.head()

,population,group,short.description,year_label,percent_raw,year,percent_value,trend_flag_for_publishing
0,children,Total,Unmet need for GP due to cost,percent.11,4.7,2011,4.7,<NA>
1,children,Boys,Unmet need for GP due to cost,percent.11,5.0,2011,5.0,<NA>
2,children,Girls,Unmet need for GP due to cost,percent.11,4.4,2011,4.4,<NA>
3,children,0-4,Unmet need for GP due to cost,percent.11,2.1,2011,2.1,<NA>
4,children,'5-9,Unmet need for GP due to cost,percent.11,6.8,2011,6.8,<NA>


### Add Indicator Metadata

The indicator mapping table is joined to add indicator ID, short name, theme and barrier type.

This keeps the trend table consistent with the other processed fact tables.

In [23]:
fact_trend = fact_trend.rename(columns={
    "short.description": "indicator_name"
})

fact_trend = fact_trend.merge(
    indicator_mapping[
        [
            "indicator_id",
            "indicator_name",
            "indicator_short_name",
            "theme",
            "barrier_type"
        ]
    ],
    on="indicator_name",
    how="left"
)

fact_trend.head()

,population,group,indicator_name,year_label,percent_raw,year,percent_value,trend_flag_for_publishing,indicator_id,indicator_short_name,theme,barrier_type
0,children,Total,Unmet need for GP due to cost,percent.11,4.7,2011,4.7,<NA>,2,GP cost barrier,Health Access,Cost
1,children,Boys,Unmet need for GP due to cost,percent.11,5.0,2011,5.0,<NA>,2,GP cost barrier,Health Access,Cost
2,children,Girls,Unmet need for GP due to cost,percent.11,4.4,2011,4.4,<NA>,2,GP cost barrier,Health Access,Cost
3,children,0-4,Unmet need for GP due to cost,percent.11,2.1,2011,2.1,<NA>,2,GP cost barrier,Health Access,Cost
4,children,'5-9,Unmet need for GP due to cost,percent.11,6.8,2011,6.8,<NA>,2,GP cost barrier,Health Access,Cost


### Reorder Columns

Columns are reordered to make the table easier to read and easier to load into PostgreSQL or Power BI.

In [24]:
fact_trend = fact_trend[
    [
        "indicator_id",
        "indicator_name",
        "indicator_short_name",
        "theme",
        "barrier_type",
        "population",
        "group",
        "year",
        "percent_value",
        "trend_flag_for_publishing",
        "percent_raw"
    ]
]

fact_trend.head()

,indicator_id,indicator_name,indicator_short_name,theme,barrier_type,population,group,year,percent_value,trend_flag_for_publishing,percent_raw
0,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,Total,2011,4.7,<NA>,4.7
1,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,Boys,2011,5.0,<NA>,5.0
2,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,Girls,2011,4.4,<NA>,4.4
3,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,0-4,2011,2.1,<NA>,2.1
4,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,children,'5-9,2011,6.8,<NA>,6.8


### Validate Trend Table

The processed trend table is validated before saving.

The validation checks whether:
- row counts are reasonable after reshaping
- all records match the indicator mapping table
- only the seven selected indicators are included
- year values are correctly created
- duplicate trend records are not created
- numeric percentage values are correctly extracted

In [25]:
print("Original trend rows:", trend.shape[0])
print("Filtered trend rows:", trend_filtered.shape[0])
print("Cleaned fact_trend rows:", fact_trend.shape[0])

Original trend rows: 9041
Filtered trend rows: 532
Cleaned fact_trend rows: 7448


In [26]:
print("Missing indicator_id:", fact_trend["indicator_id"].isna().sum())
print("Number of indicators:", fact_trend["indicator_name"].nunique())
print("Year range:", fact_trend["year"].min(), "to", fact_trend["year"].max())

Missing indicator_id: 0
Number of indicators: 7
Year range: 2011 to 2024


In [27]:
duplicate_count = fact_trend.duplicated(
    subset=[
        "indicator_id",
        "population",
        "group",
        "year"
    ]
).sum()

duplicate_count

0

In [28]:
fact_trend["trend_flag_for_publishing"].value_counts(dropna=False)

trend_flag_for_publishing
<NA>    6358
e       1042
S         48
Name: count, dtype: int64

### Save Processed Trend Table

After validation, the cleaned trend table is saved to the processed data folder as `fact_trend.csv`.

In [29]:
fact_trend.to_csv(
    processed_path / "fact_trend.csv",
    index=False
)

print("fact_trend.csv saved successfully.")

fact_trend.csv saved successfully.


## Build Dimension Tables

This section creates dimension tables for the reporting model.

Fact tables store numeric measurements, such as prevalence values, adjusted rate ratios and trend percentages.

Dimension tables store descriptive information, such as indicator names, barrier types and population groups.

These tables make the dataset easier to query in PostgreSQL and easier to model in Power BI.

### Build Indicator Dimension

The indicator dimension is created from the indicator mapping table.

It provides one row for each selected health access indicator, including the indicator ID, short name, theme and barrier type.

This table can be joined to fact tables using `indicator_id`.

In [30]:
dim_indicator = indicator_mapping[
    [
        "indicator_id",
        "indicator_name",
        "indicator_short_name",
        "theme",
        "barrier_type",
        "include_flag"
    ]
].copy()

dim_indicator

,indicator_id,indicator_name,indicator_short_name,theme,barrier_type,include_flag
0,1,Unfilled prescription due to cost,Prescription cost barrier,Health Access,Cost,True
1,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,True
2,3,Unmet need for GP due to transport,GP transport barrier,Health Access,Transport,True
3,4,Unmet need for GP due to wait time,GP wait time barrier,Health Access,Timeliness,True
4,5,Unmet need for GP due to work,GP work constraint,Health Access,Time constraint,True
5,6,Unmet need for dental health care due to cost,Dental cost barrier,Health Access,Cost,True
6,7,Unmet need for mental health care and addictio...,Mental health access barrier,Health Access,Mental health access,True


### Validate Indicator Dimension

The indicator dimension is checked to confirm that:
- it contains only the selected seven indicators
- each indicator ID is unique
- no key fields are missing 

In [31]:
print("dim_indicator rows:", dim_indicator.shape[0])
print("Unique indicator IDs:", dim_indicator["indicator_id"].nunique())
print("Unique indicator names:", dim_indicator["indicator_name"].nunique())

dim_indicator rows: 7
Unique indicator IDs: 7
Unique indicator names: 7


In [32]:
dim_indicator[
    [
        "indicator_id",
        "indicator_name",
        "indicator_short_name",
        "theme",
        "barrier_type"
    ]
].isna().sum()

indicator_id            0
indicator_name          0
indicator_short_name    0
theme                   0
barrier_type            0
dtype: int64

In [33]:
dim_indicator.duplicated(subset=["indicator_id"]).sum()

0

### Save Indicator Dimension

After validation, the indicator dimension is saved as `dim_indicator.csv`.

In [36]:
dim_indicator.to_csv(
    processed_path / "dim_indicator.csv",
    index=False
)

print("dim_indicator.csv saved successfully.")

dim_indicator.csv saved successfully.


### Build Population Group Dimension

The population group dimension is created by collecting population and group values from the processed fact tables.

This table provides a consistent list of population groups used across prevalence, subgroup comparison and trend reporting.

In [37]:
population_groups_prevalence = fact_prevalence[
    ["population", "group"]
].copy()

population_groups_trend = fact_trend[
    ["population", "group"]
].copy()

In [38]:
dim_population_group = pd.concat(
    [
        population_groups_prevalence,
        population_groups_trend
    ],
    ignore_index=True
).drop_duplicates()

dim_population_group.head()

,population,group
0,adults,Total
14,adults,15-24
15,adults,25-34
16,adults,35-44
17,adults,45-54


In [39]:
dim_population_group = dim_population_group.sort_values(
    ["population", "group"]
).reset_index(drop=True)

dim_population_group.insert(
    0,
    "population_group_id",
    range(1, len(dim_population_group) + 1)
)

dim_population_group.head()

,population_group_id,population,group
0,1,adults,15-24
1,2,adults,25-34
2,3,adults,35-44
3,4,adults,45-54
4,5,adults,55-64


### Add Population Group Category

A simple group category is added to support clearer filtering and dashboard design.

This categorises groups into broad types such as total, age group, ethnicity, deprivation, disability, gender and region.

In [40]:
def classify_group(group):
    group_text = str(group)

    if group_text == "Total":
        return "Total"

    if group_text in ["Male", "Female", "Men", "Women", "Boys", "Girls"]:
        return "Gender"

    if group_text in [
        "Māori",
        "Pacific",
        "Asian",
        "European/Other",
        "Total Māori",
        "Total Pacific",
        "Total Asian",
        "Total European/Other"
    ]:
        return "Ethnicity"

    if "Quintile" in group_text:
        return "Deprivation"

    if "Disabled" in group_text or "Non-disabled" in group_text:
        return "Disability"

    if any(region in group_text for region in [
        "Northern",
        "Midland",
        "Central",
        "South Island",
        "Te Tai Tokerau",
        "Te Manawa Taki",
        "Te Ikaroa",
        "Te Waipounamu"
    ]):
        return "Region"

    if any(age_marker in group_text for age_marker in [
        "0-4", "5-9", "10-14", "15-24", "25-34",
        "35-44", "45-54", "55-64", "65-74", "75+",
        "'1-4", "'2-4", "'5-9", "'10-14"
    ]):
        return "Age group"

    return "Other"

In [41]:
dim_population_group["group_category"] = dim_population_group["group"].apply(classify_group)

dim_population_group.head()

,population_group_id,population,group,group_category
0,1,adults,15-24,Age group
1,2,adults,25-34,Age group
2,3,adults,35-44,Age group
3,4,adults,45-54,Age group
4,5,adults,55-64,Age group


### Validate Population Group Dimension

The population group dimension is checked to confirm that:
- each population and group combination appears once
- no key fields are missing
- group categories are assigned

In [42]:
print("dim_population_group rows:", dim_population_group.shape[0])
print("Unique population-group pairs:", dim_population_group[["population", "group"]].drop_duplicates().shape[0])

dim_population_group rows: 108
Unique population-group pairs: 108


In [43]:
dim_population_group[
    [
        "population_group_id",
        "population",
        "group",
        "group_category"
    ]
].isna().sum()

population_group_id    0
population             0
group                  0
group_category         0
dtype: int64

In [44]:
dim_population_group.duplicated(
    subset=["population", "group"]
).sum()

0

In [45]:
dim_population_group["group_category"].value_counts()

group_category
Region         32
Ethnicity      16
Other          16
Disability     16
Age group      12
Deprivation    10
Gender          4
Total           2
Name: count, dtype: int64

### Save Population Group Dimension

After validation, the population group dimension is saved as `dim_population_group.csv`.

In [47]:
dim_population_group.to_csv(
    processed_path / "dim_population_group.csv",
    index=False
)

print("dim_population_group.csv saved successfully.")

dim_population_group.csv saved successfully.
